# Desafio 1 — MLP enxuta em PyTorch

Este notebook implementa um pipeline completo e reproduzível para **classificação de diabetes** e **regressão de preços de casas**. A mesma MLP simples é comparada em cinco experimentos controlados por problema.

## 1. Ambiente e reprodutibilidade

As bibliotecas abaixo são as mínimas identificadas para dados tabulares, treinamento, métricas, gráficos e TensorBoard.

In [ ]:
# Importa somente as bibliotecas usadas no pipeline.
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             mean_absolute_error, mean_squared_error, precision_score,
                             r2_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path.cwd()
DOWNLOADS = Path.home() / 'Downloads'
DIABETES_CSV = DOWNLOADS / 'diabetes_prediction' / 'diabetes_prediction_dataset.csv'
HOUSE_CSV = DOWNLOADS / 'house_price_regression' / 'house_price_regression_dataset.csv'
CHECKPOINT_DIR = ROOT / 'checkpoints'
RUN_DIR = ROOT / 'runs'
CHECKPOINT_DIR.mkdir(exist_ok=True)
RUN_DIR.mkdir(exist_ok=True)

print(f'Dispositivo: {DEVICE}')
print(f'PyTorch: {torch.__version__} | scikit-learn: {sklearn.__version__}')
assert DIABETES_CSV.exists(), f'Arquivo não encontrado: {DIABETES_CSV}'
assert HOUSE_CSV.exists(), f'Arquivo não encontrado: {HOUSE_CSV}'

In [ ]:
# Fixa todas as fontes de aleatoriedade para tornar comparações reproduzíveis.
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# Escreve um evento simples para validar TensorBoard antes de qualquer treinamento.
smoke_writer = SummaryWriter(RUN_DIR / 'ambiente')
smoke_writer.add_scalar('ambiente/verificacao', 1, 0)
smoke_writer.close()
print(f'Abra o TensorBoard com: tensorboard --logdir {RUN_DIR}')

## 2. Funções reutilizáveis

As funções abaixo mantêm o loop explícito e curto: elas criam lotes, calculam métricas, treinam/validam por época e persistem o melhor `state_dict`.

In [ ]:
# Converte arrays já transformados em DataLoaders reproduzíveis.
def make_loader(x, y, batch_size, shuffle):
    dataset = TensorDataset(torch.tensor(x, dtype=torch.float32),
                           torch.tensor(y, dtype=torch.float32))
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator)

# Mede a norma global dos gradientes para diagnosticar vanishing ou exploding gradients.
def gradient_norm(model):
    squares = [p.grad.detach().norm(2).item() ** 2 for p in model.parameters() if p.grad is not None]
    return float(np.sqrt(sum(squares))) if squares else 0.0

# Calcula as métricas de classificação a partir de logits e threshold fixado na validação.
def classification_metrics(y_true, logits, threshold=0.5):
    probabilities = 1 / (1 + np.exp(-logits.ravel()))
    predicted = (probabilities >= threshold).astype(int)
    truth = y_true.ravel().astype(int)
    return {
        'accuracy': accuracy_score(truth, predicted),
        'precision': precision_score(truth, predicted, zero_division=0),
        'recall': recall_score(truth, predicted, zero_division=0),
        'f1': f1_score(truth, predicted, zero_division=0),
    }

# Converte a regressão para a escala original antes de calcular métricas interpretáveis.
def regression_metrics(y_true_scaled, prediction_scaled, target_scaler):
    y_true = target_scaler.inverse_transform(y_true_scaled.reshape(-1, 1)).ravel()
    prediction = target_scaler.inverse_transform(prediction_scaled.reshape(-1, 1)).ravel()
    return {
        'mae': mean_absolute_error(y_true, prediction),
        'rmse': mean_squared_error(y_true, prediction) ** 0.5,
        'r2': r2_score(y_true, prediction),
    }

# Busca, somente na validação, o threshold que maximiza F1 para o problema desbalanceado.
def best_f1_threshold(y_true, logits):
    candidates = np.arange(0.10, 0.91, 0.01)
    scores = [classification_metrics(y_true, logits, t)['f1'] for t in candidates]
    return float(candidates[int(np.argmax(scores))])

In [ ]:
# Define uma MLP pequena e explícita; cada bloco é Linear -> BatchNorm opcional -> ativação -> Dropout opcional.
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, activation='relu', batchnorm=False, dropout=0.0, init='default'):
        super().__init__()
        activation_layer = nn.ReLU if activation == 'relu' else nn.LeakyReLU
        layers, previous = [], input_dim
        for width in (32, 16):
            layers.append(nn.Linear(previous, width))
            if batchnorm:
                layers.append(nn.BatchNorm1d(width))
            layers.append(activation_layer())
            if dropout:
                layers.append(nn.Dropout(dropout))
            previous = width
        layers.append(nn.Linear(previous, output_dim))
        self.network = nn.Sequential(*layers)
        if init == 'kaiming':
            for layer in self.modules():
                if isinstance(layer, nn.Linear):
                    nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
                    nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.network(x)

# Executa treino ou avaliação, retornando logits/previsões e estatísticas de gradiente por época.
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    losses, targets, outputs, norms = [], [], [], []
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            output = model(x_batch)
            loss = criterion(output, y_batch)
            if training:
                loss.backward()
                norms.append(gradient_norm(model))
                optimizer.step()
        losses.append(loss.item() * len(x_batch))
        targets.append(y_batch.detach().cpu().numpy())
        outputs.append(output.detach().cpu().numpy())
    return (sum(losses) / len(loader.dataset), np.vstack(targets), np.vstack(outputs),
            float(np.mean(norms)) if norms else 0.0, float(np.max(norms)) if norms else 0.0)

In [ ]:
# Prepara o Diabetes sem vazamento: o preprocessor é ajustado exclusivamente nas linhas de treino.
def prepare_diabetes():
    frame = pd.read_csv(DIABETES_CSV).drop_duplicates().dropna()
    x, y = frame.drop(columns='diabetes'), frame['diabetes'].to_numpy()
    x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.30, stratify=y, random_state=SEED)
    x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
    numeric = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'hypertension', 'heart_disease']
    categorical = ['gender', 'smoking_history']
    transformer = ColumnTransformer([
        ('numeric', StandardScaler(), numeric),
        ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
    ])
    x_train = transformer.fit_transform(x_train).astype(np.float32)
    x_val = transformer.transform(x_val).astype(np.float32)
    x_test = transformer.transform(x_test).astype(np.float32)
    positive_weight = (len(y_train) - y_train.sum()) / y_train.sum()
    baseline = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
    baseline.fit(x_train, y_train)
    baseline_metrics = classification_metrics(y_test, baseline.decision_function(x_test))
    return {
        'name': 'diabetes', 'input_dim': x_train.shape[1], 'batch_size': 256, 'transformer': transformer,
        'train': make_loader(x_train, y_train.reshape(-1, 1), 256, True),
        'val': make_loader(x_val, y_val.reshape(-1, 1), 256, False),
        'test': make_loader(x_test, y_test.reshape(-1, 1), 256, False),
        'pos_weight': float(positive_weight), 'baseline': baseline_metrics
    }

diabetes = prepare_diabetes()
print('Baseline Diabetes no teste:', diabetes['baseline'])

In [ ]:
# Prepara House Price sem vazamento e preserva um scaler do alvo para métricas na moeda original.
def prepare_house_price():
    frame = pd.read_csv(HOUSE_CSV).drop_duplicates().dropna()
    x, y = frame.drop(columns='House_Price'), frame['House_Price'].to_numpy().reshape(-1, 1)
    x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.30, random_state=SEED)
    x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.50, random_state=SEED)
    x_scaler, y_scaler = StandardScaler(), StandardScaler()
    x_train = x_scaler.fit_transform(x_train).astype(np.float32)
    x_val, x_test = x_scaler.transform(x_val).astype(np.float32), x_scaler.transform(x_test).astype(np.float32)
    y_train_scaled = y_scaler.fit_transform(y_train).astype(np.float32)
    y_val_scaled, y_test_scaled = y_scaler.transform(y_val).astype(np.float32), y_scaler.transform(y_test).astype(np.float32)
    baseline = LinearRegression().fit(x_train, y_train_scaled.ravel())
    baseline_metrics = regression_metrics(y_test_scaled, baseline.predict(x_test), y_scaler)
    return {
        'name': 'house_price', 'input_dim': x_train.shape[1], 'batch_size': 64, 'target_scaler': y_scaler,
        'train': make_loader(x_train, y_train_scaled, 64, True),
        'val': make_loader(x_val, y_val_scaled, 64, False),
        'test': make_loader(x_test, y_test_scaled, 64, False),
        'baseline': baseline_metrics
    }

house_price = prepare_house_price()
print('Baseline House Price no teste:', house_price['baseline'])

## 3. Experimentos controlados

Cada configuração modifica somente um aspecto da arquitetura. O checkpoint é escolhido por menor loss de validação; o vencedor do conjunto é escolhido por F1 de validação (Diabetes) ou RMSE de validação (House Price).

In [ ]:
# Declara cinco comparações pequenas e diretamente relacionadas aos requisitos do desafio.
EXPERIMENTS = [
    {'name': 'baseline', 'activation': 'relu', 'init': 'default', 'batchnorm': False, 'dropout': 0.0},
    {'name': 'leaky_relu', 'activation': 'leaky_relu', 'init': 'default', 'batchnorm': False, 'dropout': 0.0},
    {'name': 'kaiming', 'activation': 'relu', 'init': 'kaiming', 'batchnorm': False, 'dropout': 0.0},
    {'name': 'batchnorm', 'activation': 'relu', 'init': 'default', 'batchnorm': True, 'dropout': 0.0},
    {'name': 'dropout', 'activation': 'relu', 'init': 'default', 'batchnorm': True, 'dropout': 0.2},
]

# Salva somente o melhor state_dict e os metadados necessários para reconstruir o modelo.
def save_checkpoint(path, model, config, epoch, val_loss, threshold=0.5):
    torch.save({'model_state_dict': model.state_dict(), 'config': config, 'epoch': epoch,
                'best_val_loss': val_loss, 'threshold': threshold}, path)

# Treina uma configuração, registra o TensorBoard e devolve seu histórico de validação.
def train_experiment(data, config, epochs=30):
    set_seed()
    problem = data['name']
    model = MLP(data['input_dim'], 1, **{k: config[k] for k in ('activation', 'batchnorm', 'dropout', 'init')}).to(DEVICE)
    criterion = (nn.BCEWithLogitsLoss(pos_weight=torch.tensor([data['pos_weight']], device=DEVICE))
                 if problem == 'diabetes' else nn.MSELoss())
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    writer = SummaryWriter(RUN_DIR / f'{problem}_{config["name"]}')
    checkpoint = CHECKPOINT_DIR / f'{problem}_{config["name"]}.pt'
    history, best_loss = [], float('inf')
    for epoch in range(1, epochs + 1):
        train_loss, train_y, train_out, grad_mean, grad_max = run_epoch(model, data['train'], criterion, optimizer)
        val_loss, val_y, val_out, _, _ = run_epoch(model, data['val'], criterion)
        train_metrics = (classification_metrics(train_y, train_out) if problem == 'diabetes'
                         else regression_metrics(train_y, train_out, data['target_scaler']))
        val_metrics = (classification_metrics(val_y, val_out) if problem == 'diabetes'
                       else regression_metrics(val_y, val_out, data['target_scaler']))
        lr = optimizer.param_groups[0]['lr']
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                        'train_metrics': train_metrics, 'val_metrics': val_metrics,
                        'grad_mean': grad_mean, 'grad_max': grad_max, 'lr': lr})
        for prefix, values in [('train', train_metrics), ('val', val_metrics)]:
            writer.add_scalar(f'{prefix}/loss', train_loss if prefix == 'train' else val_loss, epoch)
            for key, value in values.items(): writer.add_scalar(f'{prefix}/{key}', value, epoch)
        writer.add_scalar('training/gradient_norm_mean', grad_mean, epoch)
        writer.add_scalar('training/gradient_norm_max', grad_max, epoch)
        writer.add_scalar('training/learning_rate', lr, epoch)
        scheduler.step(val_loss)
        if val_loss < best_loss:
            best_loss = val_loss
            save_checkpoint(checkpoint, model, config, epoch, val_loss)
    writer.close()
    payload = torch.load(checkpoint, map_location=DEVICE, weights_only=False)
    model.load_state_dict(payload['model_state_dict'])
    _, val_y, val_out, _, _ = run_epoch(model, data['val'], criterion)
    threshold = best_f1_threshold(val_y, val_out) if problem == 'diabetes' else 0.5
    if problem == 'diabetes':
        payload['threshold'] = threshold
        torch.save(payload, checkpoint)
    val_metrics = (classification_metrics(val_y, val_out, threshold) if problem == 'diabetes'
                   else regression_metrics(val_y, val_out, data['target_scaler']))
    return {'config': config, 'history': history, 'checkpoint': checkpoint, 'val_metrics': val_metrics}

In [ ]:
# Executa os cinco experimentos por problema; esta é a célula mais demorada do notebook.
diabetes_results = [train_experiment(diabetes, config) for config in EXPERIMENTS]
house_price_results = [train_experiment(house_price, config) for config in EXPERIMENTS]

# Seleciona vencedores exclusivamente com dados de validação.
best_diabetes = max(diabetes_results, key=lambda result: result['val_metrics']['f1'])
best_house_price = min(house_price_results, key=lambda result: result['val_metrics']['rmse'])
print('Melhor Diabetes por F1 de validação:', best_diabetes['config']['name'], best_diabetes['val_metrics'])
print('Melhor House Price por RMSE de validação:', best_house_price['config']['name'], best_house_price['val_metrics'])

In [ ]:
# Plota loss, gap treino-validação e gradient norms para nomear o comportamento observado.
def plot_history(result, title):
    history = result['history']
    epochs = [row['epoch'] for row in history]
    train_loss = [row['train_loss'] for row in history]
    val_loss = [row['val_loss'] for row in history]
    grad = [row['grad_mean'] for row in history]
    lr = [row['lr'] for row in history]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(epochs, train_loss, label='treino'); axes[0].plot(epochs, val_loss, label='validação')
    axes[0].set(title='Loss e gap treino-validação', xlabel='Época', ylabel='Loss'); axes[0].legend()
    axes[1].plot(epochs, grad); axes[1].set(title='Norma média dos gradientes', xlabel='Época', ylabel='Norma')
    axes[2].plot(epochs, lr); axes[2].set(title='Learning rate', xlabel='Época', ylabel='LR')
    for axis in axes: axis.grid(alpha=0.3)
    plt.show()

# Classifica objetivamente o comportamento para orientar a conclusão após os experimentos.
def diagnose(history):
    first, last = history[0], history[-1]
    gap = last['val_loss'] - last['train_loss']
    if max(row['grad_max'] for row in history) > 100:
        return 'instabilidade: pico de gradient norm acima de 100'
    if last['val_loss'] > min(row['val_loss'] for row in history) * 1.10 and gap > 0:
        return 'overfitting: a loss de validação piorou enquanto a de treino continuou menor'
    if last['val_loss'] >= first['val_loss'] * 0.95:
        return 'underfitting ou convergência insuficiente: a loss de validação quase não reduziu'
    return 'convergência: a loss de validação reduziu sem evidência de explosão de gradientes'

for result in [best_diabetes, best_house_price]:
    plot_history(result, result['config']['name'])
    print(result['config']['name'], '->', diagnose(result['history']))

## 4. Teste final e reprodução

A célula seguinte recarrega o melhor `state_dict`. Registre aqui, após a execução, o problema observado em um experimento, a hipótese formulada antes da correção, a correção aplicada e a evidência medida nas curvas ou normas dos gradientes.

In [ ]:
# Reconstrói o modelo somente com o checkpoint para provar que as predições são reproduzíveis.
def load_model(result, data):
    payload = torch.load(result['checkpoint'], map_location=DEVICE, weights_only=False)
    config = payload['config']
    model = MLP(data['input_dim'], 1, **{k: config[k] for k in ('activation', 'batchnorm', 'dropout', 'init')}).to(DEVICE)
    model.load_state_dict(payload['model_state_dict'])
    model.eval()
    return model, payload

# Executa inferência determinística no loader de teste.
def predict(model, loader):
    outputs, targets = [], []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            outputs.append(model(x_batch.to(DEVICE)).cpu())
            targets.append(y_batch.cpu())
    return torch.vstack(targets).numpy(), torch.vstack(outputs).numpy()

diabetes_model, diabetes_payload = load_model(best_diabetes, diabetes)
diabetes_y, diabetes_logits = predict(diabetes_model, diabetes['test'])
_, diabetes_logits_again = predict(diabetes_model, diabetes['test'])
assert torch.allclose(torch.tensor(diabetes_logits), torch.tensor(diabetes_logits_again))
diabetes_test = classification_metrics(diabetes_y, diabetes_logits, diabetes_payload['threshold'])

house_model, house_payload = load_model(best_house_price, house_price)
house_y, house_prediction = predict(house_model, house_price['test'])
_, house_prediction_again = predict(house_model, house_price['test'])
assert torch.allclose(torch.tensor(house_prediction), torch.tensor(house_prediction_again))
house_test = regression_metrics(house_y, house_prediction, house_price['target_scaler'])

print('Diabetes — baseline:', diabetes['baseline'], '| MLP:', diabetes_test)
print('House Price — baseline:', house_price['baseline'], '| MLP:', house_test)

In [ ]:
# Mostra erros de classificação e formula uma hipótese baseada no padrão que apareceu no teste.
diabetes_probabilities = 1 / (1 + np.exp(-diabetes_logits.ravel()))
diabetes_predictions = (diabetes_probabilities >= diabetes_payload['threshold']).astype(int)
matrix = confusion_matrix(diabetes_y.ravel().astype(int), diabetes_predictions)
plt.figure(figsize=(5, 4))
plt.imshow(matrix, cmap='Blues')
plt.xticks([0, 1], ['Não diabetes', 'Diabetes']); plt.yticks([0, 1], ['Não diabetes', 'Diabetes'])
plt.xlabel('Classe prevista'); plt.ylabel('Classe real'); plt.title('Matriz de confusão — Diabetes')
for row in range(2):
    for col in range(2): plt.text(col, row, matrix[row, col], ha='center', va='center')
plt.colorbar(); plt.show()
print(f'Falsos negativos: {matrix[1, 0]} | Falsos positivos: {matrix[0, 1]}')
print('Hipótese a discutir: os erros podem refletir o desbalanceamento e a sobreposição de indicadores clínicos entre as classes.')

In [ ]:
# Compara preço real e previsto e inspeciona resíduos em escala monetária.
house_actual = house_price['target_scaler'].inverse_transform(house_y).ravel()
house_predicted = house_price['target_scaler'].inverse_transform(house_prediction).ravel()
residuals = house_actual - house_predicted
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(house_actual, house_predicted, alpha=0.7)
limits = [min(house_actual.min(), house_predicted.min()), max(house_actual.max(), house_predicted.max())]
axes[0].plot(limits, limits, 'r--'); axes[0].set(title='Real vs. previsto', xlabel='Preço real', ylabel='Preço previsto')
axes[1].hist(residuals, bins=20); axes[1].axvline(0, color='r', linestyle='--')
axes[1].set(title='Resíduos', xlabel='Real - previsto', ylabel='Frequência')
for axis in axes: axis.grid(alpha=0.3)
plt.show()

## Conclusão

Após executar todas as células, complete esta seção com: (1) a comparação com os baselines no teste; (2) o experimento vencedor de cada problema; (3) o fenômeno identificado nas curvas; e (4) a hipótese, correção e efeito mensurado para um problema real observado.

### Contexto histórico

O relatório local anterior indica que o baseline linear é muito forte no pequeno dataset de imóveis e que a variância pode impedir a MLP de superá-lo. Use essa informação apenas para interpretar o resultado desta execução: as escolhas continuam sendo feitas exclusivamente pela validação atual.